<a href="https://colab.research.google.com/github/ObjectMatrix/dcc/blob/main/googlePhotoFinder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git torch torchvision faiss-cpu pillow

import clip
import torch
import glob
import json
import os
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

model, preprocess = clip.load("ViT-B/32", device=device)

In [ ]:
# Cell 2 — Mount Drive (after you've uploaded your Takeout export to Drive)
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever you unzipped/uploaded your Takeout export
PHOTOS_ROOT = "/content/drive/MyDrive/Takeout/Google Photos"
INDEX_PATH = "/content/drive/MyDrive/photo_index.pt"

In [ ]:
image_paths = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.heic"):
    image_paths.extend(glob.glob(f"{PHOTOS_ROOT}/**/{ext}", recursive=True))

print(f"Found {len(image_paths)} photos")

def get_photo_year(image_path):
    json_path = image_path + ".json"
    if os.path.exists(json_path):
        try:
            with open(json_path) as f:
                meta = json.load(f)
            ts = meta.get("photoTakenTime", {}).get("timestamp")
            if ts:
                import datetime
                return datetime.datetime.fromtimestamp(int(ts)).year
        except Exception:
            pass
    return None

photo_years = [get_photo_year(p) for p in image_paths]

In [ ]:
def embed_images(paths, batch_size=64):
    all_embeddings = []
    valid_paths = []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        batch_imgs = []
        kept_paths = []
        for p in batch_paths:
            try:
                img = preprocess(Image.open(p).convert("RGB"))
                batch_imgs.append(img)
                kept_paths.append(p)
            except Exception:
                continue  # skip corrupt/unreadable files
        if not batch_imgs:
            continue
        batch_tensor = torch.stack(batch_imgs).to(device)
        with torch.no_grad():
            emb = model.encode_image(batch_tensor)
            emb = emb / emb.norm(dim=-1, keepdim=True)  # normalize
        all_embeddings.append(emb.cpu())
        valid_paths.extend(kept_paths)
        print(f"Embedded {i+len(batch_imgs)}/{len(paths)}", end="\r")
    return torch.cat(all_embeddings), valid_paths

image_embeddings, valid_paths = embed_images(image_paths)

# Keep years aligned with valid_paths
path_to_year = dict(zip(image_paths, photo_years))
valid_years = [path_to_year.get(p) for p in valid_paths]

In [ ]:
torch.save({
    "embeddings": image_embeddings,
    "paths": valid_paths,
    "years": valid_years,
}, INDEX_PATH)

print("Saved index to", INDEX_PATH)

In [ ]:
data = torch.load(INDEX_PATH)
image_embeddings, valid_paths, valid_years = data["embeddings"], data["paths"], data["years"]

def search(query, top_k=6, year_from=None, year_to=None):
    # Optional decade/year filtering
    if year_from or year_to:
        mask = [
            (y is not None) and
            (year_from is None or y >= year_from) and
            (year_to is None or y <= year_to)
            for y in valid_years
        ]
        filtered_paths = [p for p, m in zip(valid_paths, mask) if m]
        filtered_emb = image_embeddings[torch.tensor(mask)]
    else:
        filtered_paths = valid_paths
        filtered_emb = image_embeddings

    if len(filtered_paths) == 0:
        print("No photos match that date range.")
        return []

    text = clip.tokenize([query]).to(device)
    with torch.no_grad():
        text_emb = model.encode_text(text)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    sims = (filtered_emb @ text_emb.cpu().T).squeeze(-1)
    top_indices = sims.argsort(descending=True)[:top_k]
    return [filtered_paths[i] for i in top_indices]

In [ ]:
def show_results(paths):
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(4*len(paths), 4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(Image.open(p))
        ax.axis("off")
        ax.set_title(os.path.basename(p), fontsize=8)
    plt.show()

results = search("birthday party with cake")
show_results(results)

# With decade filtering:
results_2010s = search("beach vacation", year_from=2010, year_to=2019)
show_results(results_2010s)